# Track 3 — Classify Disaster Tweets

> **REFERENCE EXAMPLE — DO NOT SUBMIT UNCHANGED**
>
> This completed notebook demonstrates one reusable workflow. Run it to understand the decisions, then choose **a different dataset** for your final project and update the target or output, features, preprocessing, split, evaluation, and explanation to fit your data.


**Question:** Is a tweet describing a real disaster?

This reference uses a public Kaggle dataset with more than 11,000 labeled tweets. The dataset may contain profanity, offensive language, or descriptions of emergencies.

The core model is **TF-IDF plus logistic regression**, not an RNN. It is fast, interpretable, and provides a strong baseline that is easy to adapt. A small LSTM/RNN comparison appears at the end as an optional extension.

Dataset: https://www.kaggle.com/datasets/vstepanenko/disaster-tweets


In [ ]:
%pip -q install "kagglehub>=1.0.3"


In [ ]:
from pathlib import Path
import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

RANDOM_STATE = 42
dataset_dir = Path(kagglehub.dataset_download("vstepanenko/disaster-tweets"))

csv_path = None
for candidate in dataset_dir.rglob("*.csv"):
    columns = set(pd.read_csv(candidate, nrows=2).columns)
    if {"text", "target"}.issubset(columns):
        csv_path = candidate
        break

if csv_path is None:
    raise FileNotFoundError("Could not find a CSV with text and target columns")

print("Using:", csv_path)


## 1. Inspect the text and labels

One row is one tweet. The target is 1 for a real disaster and 0 otherwise. Inspect examples carefully and remember that labels can contain ambiguity or annotation mistakes.


In [ ]:
tweets = pd.read_csv(csv_path)
display(tweets.head())
print("shape:", tweets.shape)
display(tweets.isna().sum().sort_values(ascending=False).to_frame("missing"))
display(tweets["target"].value_counts(dropna=False).rename("count").to_frame())
display(tweets.groupby("target")["text"].apply(lambda values: values.sample(min(3, len(values)), random_state=RANDOM_STATE)))


## 2. Clean minimally, then split

TF-IDF already handles capitalization and token boundaries. Aggressive cleaning can remove useful clues, so we only remove missing and duplicate text.

The split is stratified so both classes keep similar proportions. We split before fitting TF-IDF so the vocabulary and inverse-document frequencies learn from training text only.


In [ ]:
clean = (
    tweets
    .dropna(subset=["text", "target"])
    .drop_duplicates(subset=["text"])
    .copy()
)
clean["target"] = clean["target"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    clean["text"], clean["target"],
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=clean["target"],
)

print("train labels:")
display(y_train.value_counts(normalize=True).rename("share").to_frame())
print("test labels:")
display(y_test.value_counts(normalize=True).rename("share").to_frame())


## 3. Establish a baseline

The most-frequent baseline always predicts the larger class. F1 is the main metric because it balances precision and recall: both false alarms and missed disasters matter.


In [ ]:
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(np.zeros((len(X_train), 1)), y_train)
baseline_predictions = baseline.predict(np.zeros((len(X_test), 1)))
print("baseline F1:", f1_score(y_test, baseline_predictions))


## 4. Build the text pipeline

TF-IDF turns words and two-word phrases into weighted numeric features. Logistic regression combines those features into a probability. The full transformation lives inside the pipeline, which makes the split honest and the workflow reusable.


In [ ]:
text_model = Pipeline(steps=[
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95,
        max_features=20_000,
        sublinear_tf=True,
    )),
    ("model", LogisticRegression(
        max_iter=1_000,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    )),
])

text_model.fit(X_train, y_train)
predictions = text_model.predict(X_test)
probabilities = text_model.predict_proba(X_test)[:, 1]

print("model F1:", f1_score(y_test, predictions))
print(classification_report(y_test, predictions, digits=3))
ConfusionMatrixDisplay.from_predictions(y_test, predictions, cmap="Blues")
plt.show()


## 5. Inspect what the model learned

Logistic-regression coefficients show phrases associated with each prediction. They are model weights, not proof that a phrase causes a disaster.


In [ ]:
vectorizer = text_model.named_steps["tfidf"]
classifier = text_model.named_steps["model"]
feature_names = np.array(vectorizer.get_feature_names_out())
weights = classifier.coef_[0]

top_disaster = pd.DataFrame({"phrase": feature_names[np.argsort(weights)[-15:][::-1]], "weight": np.sort(weights)[-15:][::-1]})
top_not_disaster = pd.DataFrame({"phrase": feature_names[np.argsort(weights)[:15]], "weight": np.sort(weights)[:15]})

display(top_disaster.style.set_caption("Strongest disaster-associated phrases"))
display(top_not_disaster.style.set_caption("Strongest non-disaster-associated phrases"))


## 6. Read the errors

Error analysis is where a project becomes more than a score. Inspect confident mistakes and look for sarcasm, missing context, ambiguous labels, or phrases used in multiple ways.


In [ ]:
errors = pd.DataFrame({
    "text": X_test.to_numpy(),
    "actual": y_test.to_numpy(),
    "predicted": predictions,
    "disaster_probability": probabilities,
})
errors = errors[errors["actual"] != errors["predicted"]].copy()
errors["confidence"] = np.where(errors["predicted"] == 1, errors["disaster_probability"], 1 - errors["disaster_probability"])
display(errors.sort_values("confidence", ascending=False).head(15))


## 7. Try one change

- use only single words instead of bigrams;
- adjust the probability threshold and compare precision/recall;
- add the keyword column with a second feature pipeline;
- adapt the workflow to reviews, support tickets, headlines, or comments;
- manually categorize 20 errors and report the most common failure pattern.


In [ ]:
# Example: compare F1 at several decision thresholds.
threshold_results = []
for threshold in [0.30, 0.40, 0.50, 0.60, 0.70]:
    threshold_predictions = (probabilities >= threshold).astype(int)
    threshold_results.append({
        "threshold": threshold,
        "F1": f1_score(y_test, threshold_predictions),
    })
pd.DataFrame(threshold_results)


## Optional extension: compare a small RNN/LSTM

An RNN reads tokens in sequence and can learn order-sensitive patterns, but it introduces more decisions: vocabulary size, sequence length, embedding dimension, recurrent units, epochs, and regularization. It can also overfit this relatively small dataset.

Run this only after the TF-IDF baseline. Keep the same train/test split and metric so the comparison is fair.


In [ ]:
# OPTIONAL: a small LSTM comparison.
# Keep this False for the core lesson. In Colab, set it to True to run the RNN.
RUN_RNN = False

if not RUN_RNN:
    print("Optional RNN skipped. Set RUN_RNN = True in Colab to compare it with the TF-IDF baseline.")
else:
    import tensorflow as tf
    from tensorflow.keras import Sequential
    from tensorflow.keras.layers import Bidirectional, Dense, Dropout, Embedding, Input, LSTM, TextVectorization

    tf.keras.utils.set_random_seed(RANDOM_STATE)
    max_tokens = 10_000
    sequence_length = 60

    vectorize = TextVectorization(
        max_tokens=max_tokens,
        output_mode="int",
        output_sequence_length=sequence_length,
    )
    vectorize.adapt(X_train.to_numpy())

    rnn = Sequential([
        Input(shape=(), dtype=tf.string),
        vectorize,
        Embedding(max_tokens, 32, mask_zero=True),
        Bidirectional(LSTM(24)),
        Dropout(0.30),
        Dense(1, activation="sigmoid"),
    ])
    rnn.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    rnn.fit(
        X_train.to_numpy(), y_train.to_numpy(),
        validation_split=0.20,
        epochs=3,
        batch_size=64,
        verbose=1,
    )

    rnn_probabilities = rnn.predict(X_test.to_numpy(), verbose=0).ravel()
    rnn_predictions = (rnn_probabilities >= 0.50).astype(int)
    print("TF-IDF logistic F1:", f1_score(y_test, predictions))
    print("LSTM/RNN F1:       ", f1_score(y_test, rnn_predictions))


## Required next step: choose your own dataset

The dataset in this notebook is only the completed example. For your final project, find another labeled-text dataset such as reviews, support tickets, headlines, messages, comments, or article categories.

Before changing code, write down:

1. the new dataset source and license;
2. what one row or example represents;
3. the new target or desired output;
4. possible leakage or unavailable information;
5. the split strategy and why it matches real use;
6. the preprocessing steps that must change;
7. the baseline and main metric; and
8. one visible result you will produce with your data.

You may reuse the overall workflow and model as a starting point. You may not use the unchanged reference notebook as the final project.


## Completion checklist

- [ ] I can explain why the split is stratified.
- [ ] I compared with a most-frequent baseline.
- [ ] I can explain TF-IDF in plain language.
- [ ] I used F1 and inspected the confusion matrix.
- [ ] I read at least ten misclassified tweets.
- [ ] I changed one pipeline or threshold decision.
- [ ] If I ran the RNN, I compared it with the same split and metric rather than assuming it was better.
